# Exercise 1 (Starter) — Data Ingestion, Validation & Row Quarantine

**Timebox:** 25 minutes
**Scenario:** StreamPulse just received a fresh customer batch in `data/incoming_batch_exercise.csv`.
Some rows have messy casing/spaces and duplicates (which you should **clean**), while a few rows contain truly impossible values (e.g., `age < 18`, negative `tenure_months`, negative `monthly_charges`, or unknown `plan_tier`) which you must **quarantine** into a separate DataFrame instead of letting them pollute model training!

### Your Tasks
1. **Task 1 (`clean_and_deduplicate`)**: Strip whitespace & normalize `plan_tier` (lowercase) and `country` (uppercase), and drop duplicate `customer_id` rows (keeping the first).
2. **Task 2 (`split_valid_and_quarantine`)**: Separate rows into `(valid_df, quarantine_df)` where valid rows satisfy:
   - `18 <= age <= 100`
   - `tenure_months >= 0`
   - `0 <= monthly_charges <= 100`
   - `plan_tier` in `{"basic", "standard", "premium"}`
   - `country` in `{"US", "IN", "UK", "CA", "DE"}`
3. **Task 3**: Run the self-check cell at the bottom and make sure every `assert` passes!


In [1]:
from pathlib import Path
import pandas as pd

BATCH_PATH = Path("data/incoming_batch_exercise.csv")
raw_batch = pd.read_csv(BATCH_PATH)

print(f"Loaded raw batch: {raw_batch.shape}")
print("Unique customer_ids:", raw_batch["customer_id"].nunique())
print("Distinct plan_tier values:", raw_batch["plan_tier"].unique())
print("Distinct country values  :", raw_batch["country"].unique())
raw_batch.head(5)


Loaded raw batch: (122, 12)
Unique customer_ids: 120
Distinct plan_tier values: ['basic' ' BASIC ' 'standard' 'premium' 'Premium' 'enterprise']
Distinct country values  : ['DE' 'IN' 'UK' 'uk' 'CA' 'US' ' in ']


,customer_id,age,plan_tier,country,payment_method,auto_pay,tenure_months,monthly_charges,login_days_30d,watch_hours_30d,support_tickets_30d,churned
0,CUST-20001,26.0,basic,DE,debit_card,1,38.0,9.53,14,15.2,0,0
1,CUST-20002,53.0,BASIC,IN,bank_transfer,0,32.0,8.85,12,25.6,1,0
2,CUST-20003,22.0,basic,UK,upi,1,29.0,9.42,14,49.0,1,0
3,CUST-20004,60.0,standard,DE,upi,0,15.0,15.95,10,10.6,0,0
4,CUST-20005,29.0,basic,uk,bank_transfer,1,5.0,10.17,25,60.5,1,0


In [3]:
# ============================================================================
# TASK 1: Clean text fields and remove duplicate customer_id rows
# ============================================================================
def clean_and_deduplicate(df: pd.DataFrame) -> pd.DataFrame:
    """Return a cleaned copy of df with normalized text and unique customer_ids."""
    cleaned = df.copy()

    # TODO 1a: Strip whitespace and lowercase 'plan_tier' (only for non-null strings)
    # Hint: cleaned["plan_tier"] = cleaned["plan_tier"].str.strip().str.lower()

    # TODO 1b: Strip whitespace and uppercase 'country' (only for non-null strings)

    # TODO 1c: Drop duplicate rows based on 'customer_id', keeping the first occurrence

    raise NotImplementedError("Complete Task 1 inside clean_and_deduplicate()")
    return cleaned


In [ ]:
# ============================================================================
# TASK 2: Validate rows and separate valid_df vs quarantine_df
# ============================================================================
ALLOWED_PLANS = {"basic", "standard", "premium"}
ALLOWED_COUNTRIES = {"US", "IN", "UK", "CA", "DE"}


def split_valid_and_quarantine(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Split df into (valid_df, quarantine_df) based on StreamPulse data contract."""
    # TODO 2: Build a boolean Series `is_valid` that is True ONLY when ALL rules pass:
    #   - age between 18 and 100 (inclusive)
    #   - tenure_months >= 0
    #   - monthly_charges between 0 and 100 (inclusive)
    #   - plan_tier is in ALLOWED_PLANS
    #   - country is in ALLOWED_COUNTRIES
    #
    # Then return:
    #   valid_df = df[is_valid].reset_index(drop=True)
    #   quarantine_df = df[~is_valid].reset_index(drop=True)

    raise NotImplementedError("Complete Task 2 inside split_valid_and_quarantine()")


In [ ]:
# ============================================================================
# SELF-CHECK TESTS — Run this cell after completing Tasks 1 and 2!
# ============================================================================
cleaned_df = clean_and_deduplicate(raw_batch)
valid_df, quarantine_df = split_valid_and_quarantine(cleaned_df)

print(f"Cleaned rows    : {len(cleaned_df)}")
print(f"Valid rows      : {len(valid_df)}")
print(f"Quarantined rows: {len(quarantine_df)}")

assert cleaned_df["customer_id"].is_unique, "customer_id must be unique after deduplication!"
assert len(cleaned_df) == 120, f"Expected 120 unique rows after deduplication, got {len(cleaned_df)}"
assert len(quarantine_df) == 4, f"Expected exactly 4 invalid rows in quarantine, got {len(quarantine_df)}"
assert len(valid_df) == 116, f"Expected 116 valid rows, got {len(valid_df)}"
assert set(valid_df["plan_tier"].unique()).issubset(ALLOWED_PLANS), "Unexpected plan_tier in valid_df!"
assert set(valid_df["country"].unique()).issubset(ALLOWED_COUNTRIES), "Unexpected country in valid_df!"
assert valid_df["age"].between(18, 100).all(), "Invalid age leaked into valid_df!"
assert (valid_df["tenure_months"] >= 0).all(), "Negative tenure leaked into valid_df!"
assert valid_df["monthly_charges"].between(0, 100).all(), "Invalid monthly_charges leaked into valid_df!"

print("\n[ALL TESTS PASSED] Great job! Inspect the 4 quarantined rows below:")
quarantine_df[["customer_id", "age", "plan_tier", "tenure_months", "monthly_charges"]]


### Bonus Stretch Task (If You Finish Early!)
Add a `failure_reason` column to `quarantine_df` explaining *which* rule each row violated (e.g., `"age_out_of_range"`, `"negative_tenure"`, `"unknown_plan_tier"`) and save the quarantined rows to `artifacts/quarantined_batch.csv`.
